In [26]:
from sklearn.cluster import HDBSCAN
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import MultiPoint
from shapely import concave_hull
from shapely.ops import unary_union
import folium



In [27]:
gdf = gpd.read_file("../data/processed/philly_trees.geojson").to_crs(epsg=2272)
gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y

In [28]:

# ---------- shared params ----------
MIN_SPECIES_COUNT = 40
MIN_TREES = 15
MIN_DENSITY = 5 / 43560     # 5 trees/acre as trees/sqft
HULL_RATIO = 0.15

# ---------- buffer-union params ----------
CONNECT_FT = 100

# ---------- HDBSCAN params ----------
HDB_MIN_CLUSTER_SIZE = 20
HDB_MIN_SAMPLES = 5


def _hull_record(sp, points_gdf):
    """Build a cluster record from a set of tree points, or None if it fails filters."""
    if len(points_gdf) < MIN_TREES:
        return None
    mp = MultiPoint(list(zip(points_gdf.geometry.x, points_gdf.geometry.y)))
    hull = concave_hull(mp, ratio=HULL_RATIO)
    if hull.is_empty or hull.area == 0:
        return None
    density = len(points_gdf) / hull.area
    if density < MIN_DENSITY:
        return None
    return {
        "scientific_name": sp,
        "tree_count": len(points_gdf),
        "area_sqft": hull.area,
        "density": density,
        "geometry": hull,
    }


def cluster_buffer_union(gdf):
    records = []
    for sp in gdf["scientific_name"].value_counts().index:
        sub = gdf[gdf["scientific_name"] == sp]
        if len(sub) < MIN_SPECIES_COUNT:
            continue
        pts = gpd.GeoDataFrame(
            sub, geometry=gpd.points_from_xy(sub["x"], sub["y"]), crs=gdf.crs
        )
        blob = unary_union(pts.geometry.buffer(CONNECT_FT / 2))
        components = list(blob.geoms) if blob.geom_type == "MultiPolygon" else [blob]
        for comp in components:
            in_comp = pts[pts.within(comp)]
            rec = _hull_record(sp, in_comp)
            if rec:
                records.append(rec)
    return gpd.GeoDataFrame(records, crs=gdf.crs)


def cluster_hdbscan(gdf):
    records = []
    for sp in gdf["scientific_name"].value_counts().index:
        sub = gdf[gdf["scientific_name"] == sp]
        if len(sub) < MIN_SPECIES_COUNT:
            continue
        coords = sub[["x", "y"]].values
        labels = HDBSCAN(
            min_cluster_size=HDB_MIN_CLUSTER_SIZE,
            min_samples=HDB_MIN_SAMPLES,
            cluster_selection_method="eom",
        ).fit_predict(coords)
        sub = sub.assign(cluster=labels)
        pts = gpd.GeoDataFrame(
            sub, geometry=gpd.points_from_xy(sub["x"], sub["y"]), crs=gdf.crs
        )
        for cid, grp in pts[pts["cluster"] != -1].groupby("cluster"):
            rec = _hull_record(sp, grp)
            if rec:
                records.append(rec)
    return gpd.GeoDataFrame(records, crs=gdf.crs)


def add_clusters_to_map(m, clusters, color, method_name):
    """Add one FeatureGroup per species, prefixed with the method name."""
    clusters_wgs = clusters.to_crs(epsg=4326)
    for sp, sp_df in clusters_wgs.groupby("scientific_name"):
        sp_df = sp_df.sort_values("tree_count", ascending=False)
        fg = folium.FeatureGroup(
            name=f"[{method_name}] {sp} — {len(sp_df)} clusters", show=False
        )
        for row in sp_df.itertuples():
            folium.GeoJson(
                row.geometry.__geo_interface__,
                style_function=lambda x, c=color: {
                    "fillColor": c, "color": c, "weight": 1.5, "fillOpacity": 0.3,
                },
                tooltip=(f"<b>{sp}</b> ({method_name})<br>"
                         f"{row.tree_count} trees<br>"
                         f"{row.area_sqft/43560:.2f} acres<br>"
                         f"{row.density*43560:.1f} trees/acre"),
            ).add_to(fg)
        fg.add_to(m)


# ---------- run both, build map ----------
buf_clusters = cluster_buffer_union(gdf)
hdb_clusters = cluster_hdbscan(gdf)

print(f"buffer-union: {len(buf_clusters)} clusters across "
      f"{buf_clusters['scientific_name'].nunique()} species")
print(f"HDBSCAN:      {len(hdb_clusters)} clusters across "
      f"{hdb_clusters['scientific_name'].nunique()} species")

m = folium.Map(location=[39.9526, -75.1652], zoom_start=12, tiles="cartodbpositron")
add_clusters_to_map(m, buf_clusters, "#2b8cbe", "buffer")   # blue
add_clusters_to_map(m, hdb_clusters, "#e6550d", "hdbscan")  # orange
folium.LayerControl(collapsed=False).add_to(m)
m.save("philly_tree_clusters.html")

/Users/prince/philly-tree-mapper/venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/prince/philly-tree-mapper/venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/prince/philly-tree-mapper/venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/prince/philly-tree-mapper/venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` t

buffer-union: 466 clusters across 74 species
HDBSCAN:      428 clusters across 44 species
